In [1]:
!apt-get update && !apt-get install -y libsndfile1 ffmpeg
!pip install librosa praat-parselmouth scipy numpy pandas scikit-learn xgboost matplotlib seaborn joblib

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,533 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,580 kB]
Get:13 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,245 kB]
Get:14 http://

In [2]:
import numpy as np
import pandas as pd
import joblib
from scipy.stats import mannwhitneyu
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Aapki uploaded CSV file ka exact path
FEATURES_CSV = "/content/mdvr_kcl_features2.csv"

MUST_KEEP = ["jitter", "shimmer", "hnr", "f0_mean", "f0_std"]
CORR_THRESHOLD = 0.90
TOP_N = 20

# Load CSV
df = pd.read_csv(FEATURES_CSV)

# Auto-detect metadata & feature columns
meta = [c for c in ["subject_id", "label", "filename"] if c in df.columns]
feature_cols = [c for c in df.columns if c not in meta]

X = df[feature_cols].copy()
median = X.median()
X = X.fillna(median)

# High correlation features filter out karna
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > CORR_THRESHOLD) and c not in MUST_KEEP]
reduced = [c for c in feature_cols if c not in to_drop]

# Statistical Significance (Mann-Whitney U Test)
sig_rows = []
for col in reduced:
    hc = df.loc[df.label == "HC", col].fillna(median[col])
    pdv = df.loc[df.label == "PD", col].fillna(median[col])
    try:
        _, p = mannwhitneyu(hc, pdv)
    except ValueError:
        p = 1.0
    sig_rows.append((col, p))
sig_df = pd.DataFrame(sig_rows, columns=["feature", "p_value"])

# Feature Importance Ranking
y = (df.label == "PD").astype(int)
scaler_tmp = StandardScaler()
Xr_s = scaler_tmp.fit_transform(X[reduced])
rf_tmp = RandomForestClassifier(n_estimators=300, random_state=42)
rf_tmp.fit(Xr_s, y)
importance = pd.Series(rf_tmp.feature_importances_, index=reduced)

combined = sig_df.copy()
combined["importance"] = combined["feature"].map(importance)
combined["p_rank"] = combined["p_value"].rank()
combined["imp_rank"] = combined["importance"].rank(ascending=False)
combined["combined_rank"] = combined["p_rank"] + combined["imp_rank"]
combined = combined.sort_values("combined_rank")

selected = combined["feature"].head(TOP_N).tolist()
for must in MUST_KEEP:
    if must in reduced and must not in selected:
        selected.append(must)

# Train XGBoost Model
X_final = df[selected].fillna(median[selected])
scaler = StandardScaler()
X_s = scaler.fit_transform(X_final)

spw = (y == 0).sum() / max(y.sum(), 1)
model = XGBClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0,
    eval_metric="logloss", random_state=42, scale_pos_weight=spw
)
model.fit(X_s, y)

# Save the 4 required joblib files
joblib.dump(model, "/content/pd_model.joblib")
joblib.dump(scaler, "/content/pd_scaler.joblib")
joblib.dump(selected, "/content/pd_model_features.joblib")
joblib.dump(median[selected], "/content/pd_model_median.joblib")

print("🚀 Success! Sabhi 4 joblib files generate ho gayi hain.")

🚀 Success! Sabhi 4 joblib files generate ho gayi hain.
